# Layer 1 — Prapemrosesan Data Ritel Riil

Tiga berkas Excel di `dataset_ril/` adalah jurnal baris kasir, bukan riwayat pelanggan.
Satu baris adalah satu barang di dalam satu bukti `NO_BKT`.

Keputusan yang dipakai:

1. Hanya `TP_TRN = JUL` yang dianggap penjualan. Mutasi, penerimaan, rebate, dan adjustment dibuang.
2. `ITEM` adalah identitas produk. Nama tampilan adalah `NAMA` yang paling sering muncul untuk kode itu.
3. `NO_BKT` adalah satu struk. Tidak ada identitas pelanggan.

Folder `notebooks/` tidak dibaca dan tidak diubah. Keluaran ditulis ke `outputs_ril/`.


In [ ]:
from pathlib import Path

import pandas as pd

def find_project_dir() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "dataset_ril" / "Januari-2017.xlsx").exists():
            return candidate
    raise FileNotFoundError("Folder proyek tidak ditemukan. Jalankan notebook dari folder proyek.")

PROJECT_DIR = find_project_dir()
DATA_DIR = PROJECT_DIR / "dataset_ril"
OUTPUT_DIR = PROJECT_DIR / "outputs_ril"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = OUTPUT_DIR / "jul_transactions.csv"

FILES = [
    "Januari-2017.xlsx",
    "Februari-2017.xlsx",
    "Maret-2017.xlsx",
]
KEEP = ["TP_TRN", "NO_BKT", "ITEM", "NAMA", "KEL", "TGL_TRANS", "JAM", "QTY", "KAS"]

print("--> [INFO] Parameter Layer 1 siap: hanya transaksi JUL, identitas produk = ITEM.")
print(f"--> [INFO] Folder proyek: {PROJECT_DIR}")


## 1. Muat penjualan dan buang selain JUL

`TGL_TRANS` tersimpan sebagai nomor serial Excel. `JAM` berformat `HH:MM:SS`.
Hari memakai penomoran Senin = 0 sampai Minggu = 6. Akhir pekan adalah Sabtu dan Minggu.


In [ ]:
print("--> [INFO] Membaca tiga berkas Excel dan menyimpan hanya baris TP_TRN = JUL...")
frames = []
for name in FILES:
    raw = pd.read_excel(
        DATA_DIR / name,
        sheet_name="INTRNA",
        usecols=KEEP,
        dtype={"ITEM": str, "NO_BKT": str, "KAS": str, "TP_TRN": str, "NAMA": str, "KEL": str, "JAM": str},
        engine="openpyxl",
    )
    raw["ITEM"] = raw["ITEM"].str.replace(r"\.0$", "", regex=True).str.strip()
    raw["NO_BKT"] = raw["NO_BKT"].str.replace(r"\.0$", "", regex=True).str.strip()
    raw["sumber"] = name
    sales = raw.loc[raw["TP_TRN"].eq("JUL")].copy()
    print(f"{name}: {len(raw):,} baris jurnal, {len(sales):,} baris JUL")
    frames.append(sales)

sales = pd.concat(frames, ignore_index=True)
if not pd.api.types.is_datetime64_any_dtype(sales["TGL_TRANS"]):
    sales["TGL_TRANS"] = pd.to_datetime(pd.to_numeric(sales["TGL_TRANS"]), unit="D", origin="1899-12-30")
else:
    sales["TGL_TRANS"] = pd.to_datetime(sales["TGL_TRANS"])
jam = pd.to_datetime(sales["JAM"].astype(str), format="%H:%M:%S", errors="coerce")
if jam.isna().any():
    jam = jam.fillna(pd.to_datetime(sales["JAM"].astype(str), errors="coerce"))
sales["hour_of_day"] = jam.dt.hour.astype("int8")
sales["day_of_week"] = sales["TGL_TRANS"].dt.dayofweek.astype("int8")
sales["is_weekend"] = sales["day_of_week"].ge(5).astype("int8")
sales["bulan"] = sales["TGL_TRANS"].dt.to_period("M").astype(str)
print("--> [INFO] Bentuk gabungan JUL:", sales.shape)
print(sales["bulan"].value_counts().sort_index())


## 2. Nama tampilan dan ekspor

Satu kode `ITEM` dapat tertulis dengan beberapa ejaan `NAMA` karena nama di kasir terpotong.
Nama tampilan adalah ejaan yang paling sering. Jika jumlahnya seri, ejaan yang lebih panjang dipakai.


In [ ]:
print("--> [INFO] Menentukan nama tampilan dari kemunculan NAMA terbanyak per ITEM...")
name_counts = (
    sales.groupby(["ITEM", "NAMA"], observed=True)
    .size()
    .reset_index(name="n")
    .sort_values(["ITEM", "n", "NAMA"], ascending=[True, False, False])
)
canonical = name_counts.drop_duplicates("ITEM")[["ITEM", "NAMA"]].rename(columns={"NAMA": "nama_tampil"})
sales = sales.merge(canonical, on="ITEM", how="left")

before = len(sales)
sales = sales.drop_duplicates(["NO_BKT", "ITEM"])
print(f"Baris ganda ITEM dalam satu struk yang dibuang: {before - len(sales):,}")

export_cols = [
    "NO_BKT", "ITEM", "nama_tampil", "KEL", "TGL_TRANS", "JAM",
    "hour_of_day", "day_of_week", "is_weekend", "bulan", "QTY", "KAS", "sumber",
]
sales[export_cols].to_csv(OUTPUT_PATH, index=False)
print("--> [INFO] Tersimpan:", OUTPUT_PATH)
print("Struk:", sales["NO_BKT"].nunique(), "| Produk:", sales["ITEM"].nunique())
print(sales.groupby("bulan")["NO_BKT"].nunique().rename("struk"))
